<a href="https://colab.research.google.com/github/bindumadamanchi3/ai-upskilling-journey/blob/main/Titanic_dataset_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# Load Titanic directly from public URL — no download needed
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

print(df.shape)         # (891, 12)
print(df.columns.tolist())
# ['PassengerId','Survived','Pclass','Name','Sex','Age',
#  'SibSp','Parch','Ticket','Fare','Cabin','Embarked']
print(df.head())

(891, 12)
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.925

In [3]:
#Full missing value audit (Easy)

# Expected output: a summary table showing each column,
# null count, percentage missing, and dtype
# Only show columns that actually have missing values

null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.any() else "No nulls remaining!")

null_pct = ((null_counts/len(df)) * 100).round(2)

missing_summary = pd.DataFrame({
    'null_count': null_counts,
    'percentage_missing': null_pct,
    'dtype': df.dtypes
})

print(missing_summary)

missing_summary = missing_summary[missing_summary['null_count'] > 0].sort_values('null_count', ascending = False)
print(missing_summary)

Age         177
Cabin       687
Embarked      2
dtype: int64
             null_count  percentage_missing    dtype
PassengerId           0                0.00    int64
Survived              0                0.00    int64
Pclass                0                0.00    int64
Name                  0                0.00   object
Sex                   0                0.00   object
Age                 177               19.87  float64
SibSp                 0                0.00    int64
Parch                 0                0.00    int64
Ticket                0                0.00   object
Fare                  0                0.00  float64
Cabin               687               77.10   object
Embarked              2                0.22   object
          null_count  percentage_missing    dtype
Cabin            687               77.10   object
Age              177               19.87  float64
Embarked           2                0.22   object


In [4]:
#Visualise missing patterns (Easy)
#For each column with missing values, print a simple text-based bar showing the percentage missing. Also show how many rows are complete (no nulls in any column).

print("Missing value profile")
print('-'*50)

for col in df.columns:
   pct = df[col].isnull().mean() * 100
   if pct > 0:
     filled = int(pct/2.5)
     empty = 40 - filled
     bar     = "∎" * filled + "\u2591" * empty
     print(f"{col:<12} | {bar} | {pct:5.1f}%")

complete = df.dropna().shape[0]
print(f"Complete rows: {complete} / {len(df)}")


Missing value profile
--------------------------------------------------
Age          | ∎∎∎∎∎∎∎░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ |  19.9%
Cabin        | ∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎∎░░░░░░░░░░ |  77.1%
Embarked     | ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ |   0.2%
Complete rows: 183 / 891


In [5]:
# Investigate missing Age patterns (Medium)
# Is Age missing randomly, or is it missing more for certain groups? Check if missing Age correlates with passenger class, sex, or survival.

# Compare average Pclass and survival rate
# between passengers WITH age and WITHOUT age
# If they differ significantly, missingness is not random

df['age_missing'] = df['Age'].isnull()

comparision = df.groupby('age_missing').agg(
    p_id = ('PassengerId', 'count'),
    average_pclass = ('Pclass', 'mean'),
    avg_survival = ('Survived', 'mean'),
    pct_male = ('Sex', lambda x: (x == 'male').mean())
).round(3)

print(comparision)

# comparision.index= ['Age Present', 'Age Missing']
# print(comparision)  #while using this, if the dataFrame rows ever sorted or flipped upside down, labels will mismatch and accidentally lie about the data! using rename is the safe bet.

# OR

comparision = comparision.rename(index={False: 'Age Present', True: 'Age Missing'})
print(comparision)

             p_id  average_pclass  avg_survival  pct_male
age_missing                                              
False         714           2.237         0.406     0.634
True          177           2.599         0.294     0.701
             p_id  average_pclass  avg_survival  pct_male
age_missing                                              
Age Present   714           2.237         0.406     0.634
Age Missing   177           2.599         0.294     0.701


In [6]:
# Investigate Cabin missingness (Medium)
# Cabin is 77% missing. Dig into why — check if Cabin missingness correlates strongly with passenger class. Then decide: is Cabin salvageable as a feature?
# Check: what % of each passenger class has Cabin info?
# Check: survival rate for those with vs without Cabin

df['cabin_present'] = df['Cabin'].isnull()

# comparision = df.groupby('cabin_missing').aggr(
#     p_class = ()
# )
pct_cabin_info = df.groupby('Pclass').agg(
    pct_has_cabin_info = ('cabin_present', lambda x : (x == True).mean() * 100)
).round(3)
print(pct_cabin_info)

survival_rate_cabin = df.groupby('cabin_present').agg(
    survival_rate = ('Survived', 'mean')
).round(3)

survival_rate_cabin = survival_rate_cabin.rename(index={False: 'Cabin Present', True: 'Cabin Missing'})
print(survival_rate_cabin)


        pct_has_cabin_info
Pclass                    
1                   18.519
2                   91.304
3                   97.556
               survival_rate
cabin_present               
Cabin Present          0.667
Cabin Missing          0.300


In [7]:
# Fill missing Age intelligently (Medium)
# Fill missing Age values using the median Age grouped by Sex and Pclass — not the overall median. Passengers of different class and sex had very different age distributions.
# Strategy: for each (Sex, Pclass) group, fill missing age
# with the median age of that specific group
# e.g. female 1st class median age ≠ male 3rd class median age

median_age = df.groupby(['Sex','Pclass'])['Age'].median()
print("Median age by Sex + Pclass:")
print(median_age)

#Filling in missed values for age with the calculated median
df['Age'] = df['Age'].fillna(df.groupby(['Sex','Pclass'])['Age'].transform('median'))
print("Checking for nulls after filling missed values:", df['Age'].isnull().sum())


Median age by Sex + Pclass:
Sex     Pclass
female  1         35.0
        2         28.0
        3         21.5
male    1         40.0
        2         30.0
        3         25.0
Name: Age, dtype: float64
Checking for nulls after filling missed values: 0


In [8]:
# Fix Embarked and drop Cabin (Easy)
# Fill the 2 missing Embarked values with the most common port. Then drop the Cabin column entirely and replace it with the has_cabin binary feature you created in Exercise 4.
# After cleaning:
# Embarked: 0 nulls
# Cabin column: gone
# has_cabin column: 1 if original Cabin was present, 0 otherwise

#To inspect or view all the values there in 'Enmarked; column
print(df['Embarked'].value_counts()) #this gives the valus with count
print(df['Embarked'].unique()) #['S' 'C' 'Q' nan] this gives the clean list of the unique names and not count

#Instead of hardcoding "S" into your script after reading the printout, you can make your code dynamic by using .mode()[0]
most_common_port = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(most_common_port)

print("Most common port is:",most_common_port)
print("Null value count for Embark column after cleaning is:", df['Embarked'].isnull().sum())

#Adding has_cabin column
df['has_cabin'] = df['cabin_present'].astype(int)

#Drop cabin column and not needed columns
df = df.drop(columns=['Cabin', 'cabin_present', 'age_missing'])

print("Columns now:", df.columns.to_list())


Embarked
S    644
C    168
Q     77
Name: count, dtype: int64
['S' 'C' 'Q' nan]
Most common port is: S
Null value count for Embark column after cleaning is: 0
Columns now: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked', 'has_cabin']


In [9]:
# Engineer new features from existing columns (Medium)
# Create useful new features that might help explain survival:

# family_size — total family members on board (SibSp + Parch + 1 for self)
# is_alone — 1 if travelling alone, 0 otherwise
# title — extract title from Name (Mr, Mrs, Miss, Master, etc.)
# age_group — bin Age into: Child (0-12), Teen (13-17), Adult (18-60), Senior (60+)

# Expected new columns: family_size, is_alone, title, age_group

def extract_title(name_string):
    # Split by comma to get the right half, then split by period to get the left half
    parts = name_string.split(',')
    title_section = parts[1].split('.')
    return title_section[0].strip()

def get_ageGroup(age):
  #Handle missing data immediately
  if pd.isna(age):
        return "Unknown"

  if 0 < age <= 12:
    return 'Child'
  elif age <= 17:
    return 'Teen'
  elif age < 60:
    return 'Adult'
  else:
    return 'Senior'

df['family_size'] = df['SibSp'] + df['Parch'] + 1

df['is_alone'] = (df['family_size'] == 1).astype(int)

# title — extract from Name using regex
df['title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
print("Raw titles:", df['title'].value_counts().head(8).to_dict()) # {'Mr': 517, 'Miss': 182, 'Mrs': 125, 'Master': 40, ...}

#OR

# df['title'] = df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip() #pandas way. pandas doesn't know native python methods. so we use str before using the python methods

#OR

# Apply the clean Python function to every row in the Name column(function above in this snippet)
# df['title'] = df['Name'].apply(extract_title)

bins = [0, 12, 17, 60, 120] #by default, right is included (0, 12] (right = True). use right=False if you want the left side value to be included and not right
labels = ['Child', 'Teen', 'Adult', 'Senior']
df['age_group'] = pd.cut(df['Age'], bins = bins, labels = labels) # pd.cut(df['Age'], bins=age_bins, labels=group_names, right=False) if you don't want the right to be included

# OR

# df['age_group'] = df['Age'].apply(get_ageGroup) # this is also correct but a bit slower when compared to pd.cut. d.cut is safe when there is null values.

print(df.head())


Raw titles: {'Mr': 517, 'Miss': 182, 'Mrs': 125, 'Master': 40, 'Dr': 7, 'Rev': 6, 'Col': 2, 'Mlle': 2}
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Embarked  has_cabin  family_size  \
0      0         A/5 21171   7.2500        S          1            2   
1      0          PC 17599  71.2833        C          0            2   
2

In [10]:
#Final clean dataset check (Easy)
# Do a full final audit of the cleaned dataset: check all nulls are gone (except acceptable ones), confirm dtypes, print a clean summary of all columns.
# After all cleaning:
# Zero nulls in: Age, Embarked, has_cabin
# Confirm numeric columns are numeric
# Print shape, null summary, and df.describe()

#print(df.info())

# # Programmatic verification assertions
# critical_columns = ['Age', 'Embarked', 'cabin_missing', 'family_size', 'is_alone', 'title', 'age_group']
# existing_crit_cols = [col for col in critical_columns if col in df.columns]
# print(existing_crit_cols)

# print("\nValidation Check:")
# has_failures = False
# for col in existing_crit_cols:
#     null_count = df[col].isnull().sum()
#     if null_count == 0:
#         print(f"  ✅ {col}: 0 missing values. Clean!")
#     else:
#         print(f"  ❌ {col}: Found {null_count} missing values!")
#         has_failures = True

# o/p
# Validation Check:
#   ❌ Age: Found 177 missing values!
#   ❌ Embarked: Found 2 missing values!

print("Cleaned Dataset Audit")
print("-" * 40)
print("Printing shape:")
print(f"{df.shape[0]} rows, {df.shape[1]} columns")

print("\nChecking for Null's:")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "No nulls remaining!")

print("\nColumn dtypes:")
print(df.dtypes)

print("\nNumeric Summary:")
print(df.describe().round(2))



Cleaned Dataset Audit
----------------------------------------
Printing shape:
891 rows, 16 columns

Checking for Null's:
No nulls remaining!

Column dtypes:
PassengerId       int64
Survived          int64
Pclass            int64
Name             object
Sex              object
Age             float64
SibSp             int64
Parch             int64
Ticket           object
Fare            float64
Embarked         object
has_cabin         int64
family_size       int64
is_alone          int64
title            object
age_group      category
dtype: object

Numeric Summary:
       PassengerId  Survived  Pclass     Age   SibSp   Parch    Fare  \
count       891.00    891.00  891.00  891.00  891.00  891.00  891.00   
mean        446.00      0.38    2.31   29.11    0.52    0.38   32.20   
std         257.35      0.49    0.84   13.30    1.10    0.81   49.69   
min           1.00      0.00    1.00    0.42    0.00    0.00    0.00   
25%         223.50      0.00    2.00   21.50    0.00    0.00    7.

In [11]:
# Overall survival rate and basic breakdown (Easy)
# Compute the overall survival rate. Then break it down by the three most important factors: Sex, Pclass, and Embarked.
# Overall survival rate
# Survival rate by Sex
# Survival rate by Pclass
# Survival rate by Embarked port

survival_rate = df['Survived'].mean().round(3) * 100
print("Overall survived rate: ", survival_rate)

print("\nSurvival rate by sex:")
survival_rate_sex = df.groupby('Sex')['Survived'].mean().round(3) * 100
print(survival_rate_sex)

print("\nSurvival rate by Pclass:")
survival_rate_pclass = df.groupby('Pclass')['Survived'].mean().round(3) * 100
print(survival_rate_pclass)

print("\nSurvival rate by Embarked:")
survival_rate_embarked = df.groupby('Embarked')['Survived'].mean().round(3) * 100
print(survival_rate_embarked)


Overall survived rate:  38.4

Survival rate by sex:
Sex
female    74.2
male      18.9
Name: Survived, dtype: float64

Survival rate by Pclass:
Pclass
1    63.0
2    47.3
3    24.2
Name: Survived, dtype: float64

Survival rate by Embarked:
Embarked
C    55.4
Q    39.0
S    33.9
Name: Survived, dtype: float64


In [12]:
# Sex + Pclass combined survival (Medium)
# The real insight comes from combining factors. Build a pivot table showing survival rate for every Sex × Pclass combination.
# Expected: a 2x3 table
# Rows: female / male
# Cols: 1st / 2nd / 3rd class
# Values: survival rate as percentage

#There are two ways to do this. But the most preferred is the second one.
#1st way
sex_pclass_survival = df.groupby(['Sex', 'Pclass'])['Survived'].mean().round(3) * 100
pivot_grid = sex_pclass_survival.unstack()
print(pivot_grid)

#2nd way
pivot = df.pivot_table(
    index = 'Sex',
    columns = 'Pclass',
    values='Survived',
    aggfunc='mean'
)*100

print("\n")
pivot.columns = ['1st Class', '2nd Class', '3rd Class'] #optional. If not used, will be similar to 1st output
print(pivot.round(1))

print(f"\nBest Group Survived is female 1st class with rate: {pivot.loc['female', '1st Class']:.1f}")
print(f"Worst group survived is male 3rd class with rate:{pivot.loc['male', '3rd Class']:.1f}")
print(f"Difference: {pivot.loc['female','1st Class'] - pivot.loc['male','3rd Class']:.1f} percentage points")


Pclass     1     2     3
Sex                     
female  96.8  92.1  50.0
male    36.9  15.7  13.5


        1st Class  2nd Class  3rd Class
Sex                                    
female       96.8       92.1       50.0
male         36.9       15.7       13.5

Best Group Survived is female 1st class with rate: 96.8
Worst group survived is male 3rd class with rate:13.5
Difference: 83.3 percentage points


In [13]:
# Age and survival (Medium)
# Analyse how age relates to survival. Compare survival rates across your age_group bins. Also compute the mean and median age of survivors vs non-survivors.
# 11a. Survival rate by age_group
# 11b. Mean and median age: survivors vs non-survivors
# 11c. What pattern do you observe?

age_survival = df.groupby('age_group', observed=True).agg(
    count         = ('Survived', 'count'),
    survival_rate = ('Survived', 'mean')
).round(3)
age_survival['survival_rate'] = age_survival['survival_rate'] *100
print("Survival by age group:")
print(age_survival)

age_survivors = df.groupby("Survived")["Age"].agg(['mean', 'median']).round(1)
age_survivors.index = ['Did not Survived', 'Survived']
print("\nMean and median age - survivors vs non-survivors:")
print(age_survivors)

print(f"\nChildren had the most survival rate: {age_survival.loc['Child', 'survival_rate']:.1f}")
print(f"Seniors had the worst survival rate: {age_survival.loc['Senior', 'survival_rate']:.1f}")
print("Age difference between survivors and non-survivors is small which is ~2 yrs")
print("Sex and Class have more predictive power than Age alone")


Survival by age group:
           count  survival_rate
age_group                      
Child         69           58.0
Teen          44           47.7
Adult        756           36.5
Senior        22           22.7

Mean and median age - survivors vs non-survivors:
                  mean  median
Did not Survived  29.7    25.0
Survived          28.1    27.0

Children had the most survival rate: 58.0
Seniors had the worst survival rate: 22.7
Age difference between survivors and non-survivors is small which is ~2 yrs
Sex and Class have more predictive power than Age alone


In [14]:
#  Family size and survival (Medium)
# Did travelling with family help or hurt survival chances? Analyse survival by family_size and is_alone.
# 12a. Survival rate by is_alone (solo vs with family)
# 12b. Survival rate by family_size (1 through max)
# 12c. Is there an optimal family size?


print("Survival rate by is_alone:")
survival_rate_alone = df.groupby('is_alone')['Survived'].mean().round(3) * 100
survival_rate_alone.index = ["With family", "Solo"]
print(survival_rate_alone)

print("\nSurvival rate by family_size:")
survival_rate_famiy = df.groupby('family_size')['Survived'].mean().round(3) * 100
print(survival_rate_famiy)

best_size = survival_rate_famiy.idxmax()
print(f"\nOptimal family size: {best_size} members")
print("Large families (7+) had 0% survival — too many to evacuate together")


Survival rate by is_alone:
With family    50.6
Solo           30.4
Name: Survived, dtype: float64

Survival rate by family_size:
family_size
1     30.4
2     55.3
3     57.8
4     72.4
5     20.0
6     13.6
7     33.3
8      0.0
11     0.0
Name: Survived, dtype: float64

Optimal family size: 4 members
Large families (7+) had 0% survival — too many to evacuate together


In [15]:
# Title and survival (Medium)
# Use the engineered title feature to analyse survival rates. Does social status (captured by title) predict survival?
# Survival rate by title
# Count and survival rate for each title group

survival_title = df.groupby('title').agg(
    count = ('Survived', 'count'),
    survival_rate = ('Survived', 'mean'),
    avg_age = ('Age', 'mean')
).round(1)
survival_title['survival_rate'] = survival_title['survival_rate'] * 100
print("Survival rate by title:")
print(survival_title.sort_values('survival_rate', ascending = False))

print(f"\nMaster avg age: {survival_title.loc['Master','avg_age']:.1f} years")
print("'Master' = young boys → explains high survival vs adult males")


Survival rate by title:
              count  survival_rate  avg_age
title                                      
Lady              1          100.0     48.0
Ms                1          100.0     28.0
Sir               1          100.0     49.0
Mme               1          100.0     24.0
the Countess      1          100.0     33.0
Mlle              2          100.0     24.0
Mrs             125           80.0     34.8
Miss            182           70.0     21.9
Master           40           60.0      6.6
Major             2           50.0     48.5
Col               2           50.0     58.0
Dr                7           40.0     41.7
Mr              517           20.0     31.3
Capt              1            0.0     70.0
Jonkheer          1            0.0     38.0
Don               1            0.0     40.0
Rev               6            0.0     43.2

Master avg age: 6.6 years
'Master' = young boys → explains high survival vs adult males


In [17]:
# Fare and survival (Medium)
# Analyse how ticket fare relates to survival. Group fares into quartiles and compare survival rates.
# 14a. Mean fare for survivors vs non-survivors
# 14b. Create fare quartile bins, compute survival rate per quartile
# 14c. Correlation between fare and survival

survival_fare = df.groupby('Survived')['Fare'].agg(['mean','median']).round(2)
survival_fare.index = ['Did not Survive', 'Survived']
print("Mean fare for survivors vs non-survivors:")
print(survival_fare)

df['fare_quartile'] = pd.qcut(df['Fare'], q=4,
                               labels=['Q1\nLowest','Q2','Q3','Q4\nHighest'])
quartile_survival = df.groupby('fare_quartile', observed=True).agg(
    count         = ('Survived', 'count'),
    survival_rate = ('Survived', 'mean'),
    avg_fare      = ('Fare',     'mean')
).round(2)
quartile_survival['survival_rate'] = (quartile_survival['survival_rate'] * 100).round(1)
print("\nSurvival by fare quartile:")
print(quartile_survival)

corr = df['Fare'].corr(df['Survived'])
print(f"\nFare-Survival correlation: {corr:.3f}")
#corr value in between -1 and 1. >0 means there is a direct relationship. As Fare goes up, Survival rates go up.
#The Strength is "Weak to Moderate": A value around 0.26 means the relationship is real and statistically significant, but it isn't a perfect predictor.
#Correlation of 0.257 is moderate — meaningful but not dominant on its own.


Mean fare for survivors vs non-survivors:
                  mean  median
Did not Survive  22.12    10.5
Survived         48.40    26.0

Survival by fare quartile:
               count  survival_rate  avg_fare
fare_quartile                                
Q1\nLowest       223           20.0      7.03
Q2               224           30.0     10.39
Q3               222           45.0     23.03
Q4\nHighest      222           58.0     88.68

Fare-Survival correlation: 0.257


In [59]:
#  COMBINED CHALLENGE (Hard)
# Build titanic_survival_report(df) that returns a complete analysis dict:

# overall_survival_pct — overall survival rate as percentage (1dp)
# best_survival_group — string like "female, class 1" with highest survival
# worst_survival_group — string like "male, class 3" with lowest survival
# children_survival_pct — survival rate for children (age <= 12)
# alone_vs_family_diff — difference in survival rate between family travellers and solo travellers (in percentage points, 1dp)
# top_predictors — list of (column, correlation_with_survival) tuples, top 3 numeric columns by absolute correlation, sorted descending

def titanic_survival_report(df):

  overall_pct = df['Survived'].mean().round(2) * 100

  best_survived_group = df.groupby(['Sex', 'Pclass'])['Survived'].mean() * 100
  best_idx = best_survived_group.idxmax()
  worst_idx = best_survived_group.idxmin()

  best_group = f"{best_idx[0]} class {best_idx[1]}"
  worst_group = f"{worst_idx[0]}, class {worst_idx[1]}"

  age_wise_pct = df.groupby('age_group')['Survived'].mean().round(1)* 100
  children_pct = age_wise_pct['Child']
  # OR
  # children      = df[df['Age'] <= 12]
  # children_pct  = round(children['Survived'].mean() * 100, 1)

  alone_vs_family = df.groupby('is_alone')['Survived'].mean() * 100
  alone_vs_family_diff = abs(alone_vs_family[0] - alone_vs_family[1]).round(1)
  #OR
  # alone_rate  = df[df['is_alone'] == 1]['Survived'].mean() * 100
  # family_rate = df[df['is_alone'] == 0]['Survived'].mean() * 100
  # alone_diff  = round(family_rate - alone_rate, 1)

  numeric_df = df.select_dtypes(include= ['number'])
  corr_series = numeric_df.corr()['Survived']
  top3_corr = corr_series.drop('Survived').abs().sort_values(ascending=False).head(3).round(1)
  top3_predictors = list(top3_corr.items())
  #OR
  # Top numeric correlations with Survived
  # numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
  # numeric_cols = [c for c in numeric_cols
  #                 if c not in ['Survived', 'PassengerId']]

  # correlations = []
  # for col in numeric_cols:
  #     corr = df[col].corr(df['Survived'])
  #     if not np.isnan(corr):
  #         correlations.append((col, round(corr, 3)))

  # top_predictors = sorted(correlations,
  #                         key=lambda x: abs(x[1]),
  #                         reverse=True)[:3]

  return {
      'overall_survival_pct' : overall_pct,
      'best_survival_group'  : best_group,
      'worst_survival_group' : worst_group,
      'children_survival_pct': children_pct,
      'alone_vs_family_diff' : alone_vs_family_diff,
      'top_predictors'       : top3_predictors
  }

report = titanic_survival_report(df)
print("\n===== TITANIC SURVIVAL REPORT =====")
for k, v in report.items():
    print(f"{k}: {v}")


===== TITANIC SURVIVAL REPORT =====
overall_survival_pct: 38.0
best_survival_group: female class 1
worst_survival_group: male, class 3
children_survival_pct: 60.0
alone_vs_family_diff: 20.2
top_predictors: [('Pclass', 0.3), ('has_cabin', 0.3), ('Fare', 0.3)]


/tmp/ipykernel_9961/2175899158.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_wise_pct = df.groupby('age_group')['Survived'].mean().round(1)* 100
